# AIMET-Enhanced MXFP4 Quantization for Visual Wake Word

This notebook loads the Visual Wake Word TFLite model, applies Qualcomm AIMET
optimizations, quantizes to MXFP4, and evaluates accuracy on COCO minival.

**Pipeline:**
1. Clone the VWW repo (contains `.h5` and `.tflite` models + minival IDs)
2. Load the VWW TFLite model (+ Keras H5 for AIMET, which needs a live graph)
3. **Cross-Layer Equalization (CLE)** — rebalances weight ranges across layers
4. **Bias Correction (BC)** — corrects quantization-induced bias shifts
5. **AdaRound** — learned rounding minimizing reconstruction error
6. **MXFP4 (E2M1)** quantization with block-wise E8M0 scales
7. Evaluate all variants on COCO minival (person/not-person detection)

**GPU not required.** CLE, Bias Correction, and MXFP4 quantization are pure weight
manipulation (CPU-only). AdaRound's gradient loop runs faster on GPU but is fine on
CPU for small models like VWW. Use **CPU runtime** to avoid Colab GPU time limits.

**AIMET compatibility note:** AIMET only ships Python 3.10 wheels (cp310), but Colab
now defaults to Python 3.12. The notebook includes standalone implementations of
CLE, Bias Correction, and AdaRound that implement the same algorithms from the
original papers and work on any Python version.

**Before running:**
1. Upload `val2014/`, `annotations/` (with `instances_val2014.json`) to Google Drive
2. Optionally upload `train2014/` for real calibration data (synthetic fallback available)
3. Update COCO data paths in Cell 2 if needed
4. Models (`modelVisualWakeWord.h5`, `.tflite`) are loaded automatically from the repo

In [ ]:
#@title 1. Mount Google Drive, Clone Repo & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

import sys, importlib, subprocess

# Clone the VWW repo — contains models (.h5, .tflite) and minival IDs
REPO_DIR = '/content/vww'
BRANCH = "claude/explain-codebase-mljz2na9k1pqxa7f-C4z0d"
!git clone -b {BRANCH} https://github.com/amitmate/visualwakeword.git {REPO_DIR} 2>/dev/null \
    || (cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH})

# Verify models exist in the repo
import os
assert os.path.isfile(f'{REPO_DIR}/modelVisualWakeWord.h5'), "H5 model not found in repo"
assert os.path.isfile(f'{REPO_DIR}/modelVisualWakeWord.tflite'), "TFLite model not found in repo"
print(f"Repo cloned to {REPO_DIR}")
!ls -lh {REPO_DIR}/modelVisualWakeWord.* {REPO_DIR}/compressed_models/*.tflite

# Check Python version — AIMET only has cp310 wheels
py_ver = f"{sys.version_info.major}.{sys.version_info.minor}"
print(f"\nPython: {py_ver}")

# Try installing AIMET (best-effort — requires Python 3.10)
AIMET_AVAILABLE = False
if py_ver == "3.10":
    print("Python 3.10 detected — attempting AIMET install...")
    !pip install -q https://github.com/quic/aimet/releases/download/1.34.0/aimet_tensorflow-1.34.0.cpu-cp310-cp310-manylinux_2_34_x86_64.whl 2>&1 | tail -3
    try:
        importlib.import_module('aimet_tensorflow')
        AIMET_AVAILABLE = True
    except ImportError:
        pass
else:
    print(f"Python {py_ver} — AIMET wheels only support 3.10 (cp310).")
    print("Trying pip install anyway in case a compatible wheel exists...")
    !pip install -q aimet-tensorflow 2>/dev/null && echo "Success" || echo "Not available"
    try:
        importlib.import_module('aimet_tensorflow')
        AIMET_AVAILABLE = True
    except ImportError:
        pass

!pip install -q pycocotools opencv-python-headless

import struct, gzip
import numpy as np
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import cv2
from pycocotools.coco import COCO

if AIMET_AVAILABLE:
    import aimet_tensorflow
    print(f"\nAIMET: {aimet_tensorflow.__version__}")
else:
    print(f"\nAIMET not available (needs Python 3.10, got {py_ver}).")
    print("Using standalone CLE + Bias Correction + AdaRound implementations.")
    print("These implement the same algorithms from the original papers.")

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU') or 'None (CPU only — this is fine)'}")

In [ ]:
#@title 2. Configure Paths { display-mode: "form" }

# ─── Models come from the cloned repo ────────────────────────────────────
h5_model_path = f'{REPO_DIR}/modelVisualWakeWord.h5'
tflite_model_path = f'{REPO_DIR}/modelVisualWakeWord.tflite'
minival_path = f'{REPO_DIR}/mscocominival.txt'

#@markdown **COCO data paths (Google Drive):**
val2014_folder = "/content/drive/MyDrive/val2014" #@param {type:"string"}
annotations_folder = "/content/drive/MyDrive/annotations" #@param {type:"string"}
train2014_folder = "/content/drive/MyDrive/train2014" #@param {type:"string"}

#@markdown **Settings:**
input_size = 96 #@param {type:"integer"}
n_calibration = 200 #@param {type:"integer"}
adaround_iters = 200 #@param {type:"integer"}

INPUT_SHAPE = (input_size, input_size, 3)
IMAGESIZE = input_size
BLOCK_SIZE = 32
PERSONTHR = 2048

# Verify model files from repo
assert os.path.isfile(h5_model_path), f"H5 model not found: {h5_model_path}"
assert os.path.isfile(tflite_model_path), f"TFLite model not found: {tflite_model_path}"
assert os.path.isfile(minival_path), f"minival IDs not found: {minival_path}"

# Verify COCO data on Drive
assert os.path.isdir(val2014_folder), f"val2014 not found: {val2014_folder}"
ann_file = os.path.join(annotations_folder, 'instances_val2014.json')
assert os.path.isfile(ann_file), f"Annotations not found: {ann_file}"

has_train = os.path.isdir(train2014_folder)

h5_size = os.path.getsize(h5_model_path)
tflite_size = os.path.getsize(tflite_model_path)
print(f"Models (from repo):")
print(f"  H5 model:     {h5_model_path}  ({h5_size/1024:.1f} KB)")
print(f"  TFLite model: {tflite_model_path}  ({tflite_size/1024:.1f} KB)")
print(f"  Minival IDs:  {minival_path}")
print(f"\nCOCO data (from Drive):")
print(f"  val2014:      {val2014_folder}")
print(f"  annotations:  {annotations_folder}")
print(f"  train2014:    {'real images from ' + train2014_folder if has_train else 'NOT FOUND — will use synthetic calibration'}")
print("\nAll paths verified.")

In [ ]:
#@title 3. Load VWW TFLite Model & Verify

# Load and inspect the TFLite model
interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("=== VWW TFLite Model ===")
print(f"Input:  shape={input_details[0]['shape']}, dtype={input_details[0]['dtype'].__name__}")
print(f"Output: shape={output_details[0]['shape']}, dtype={output_details[0]['dtype'].__name__}")

if input_details[0]['dtype'] == np.int8:
    iq = input_details[0]['quantization_parameters']
    print(f"Input quant:  scale={iq['scales'][0]:.6f}, zero_point={iq['zero_points'][0]}")
if output_details[0]['dtype'] == np.int8:
    oq = output_details[0]['quantization_parameters']
    print(f"Output quant: scale={oq['scales'][0]:.6f}, zero_point={oq['zero_points'][0]}")

# Also load the Keras model (needed for AIMET — it requires a live computation graph)
print("\nLoading Keras H5 model (needed for AIMET graph manipulation)...")
model_orig = tf.keras.models.load_model(h5_model_path, compile=False)
model_orig.summary()

# Quick sanity check: both models should agree on a test input
test_input = np.random.RandomState(0).randn(1, input_size, input_size, 3).astype(np.float32) * 0.5
h5_pred = model_orig.predict(test_input, verbose=0)[0, 0]

if input_details[0]['dtype'] == np.int8:
    iscale = input_details[0]['quantization_parameters']['scales'][0]
    izp = input_details[0]['quantization_parameters']['zero_points'][0]
    tfl_input = (test_input / iscale + izp).astype(np.int8)
else:
    tfl_input = test_input

interpreter.set_tensor(input_details[0]['index'], tfl_input)
interpreter.invoke()
tfl_out = interpreter.get_tensor(output_details[0]['index'])
if output_details[0]['dtype'] == np.int8:
    oscale = output_details[0]['quantization_parameters']['scales'][0]
    ozp = output_details[0]['quantization_parameters']['zero_points'][0]
    tfl_pred = (float(tfl_out[0, 0]) - ozp) * oscale
else:
    tfl_pred = float(tfl_out[0, 0])

print(f"\nSanity check — H5 pred: {h5_pred:.4f}, TFLite pred: {tfl_pred:.4f}")
print("Models loaded successfully.")

In [ ]:
#@title 4. Build Calibration Dataset

def load_calibration_data(cal_dir, n_samples, img_size):
    """Load calibration images for AdaRound + TFLite conversion."""
    images = []
    if cal_dir and os.path.isdir(cal_dir):
        files = sorted([f for f in os.listdir(cal_dir)
                        if f.lower().endswith(('.jpg', '.png'))])
        rng = np.random.RandomState(42)
        rng.shuffle(files)
        for fname in files[:n_samples]:
            img = cv2.imread(os.path.join(cal_dir, fname))
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            h, w = img.shape[:2]
            side = max(h, w)
            canvas = np.zeros((side, side, 3), dtype=img.dtype)
            y0, x0 = (side - h) // 2, (side - w) // 2
            canvas[y0:y0+h, x0:x0+w] = img
            img = cv2.resize(canvas, (img_size, img_size), interpolation=cv2.INTER_AREA)
            img = img.astype(np.float32) / 127.5 - 1.0
            images.append(img)
        print(f"Loaded {len(images)} real calibration images from {cal_dir}")
    else:
        print("Using synthetic calibration data (upload train2014 for better results)")
        rng = np.random.RandomState(42)
        mean = np.array([-0.030, -0.088, -0.188], dtype=np.float32)
        std = np.array([0.458, 0.448, 0.450], dtype=np.float32)
        for _ in range(n_samples):
            img = rng.randn(img_size, img_size, 3).astype(np.float32) * std + mean
            images.append(np.clip(img, -1.0, 1.0))
    return np.array(images)

cal_data = load_calibration_data(train2014_folder, n_calibration, input_size)
print(f"Calibration data shape: {cal_data.shape}")

In [ ]:
#@title 5. MXFP4 Codec — E2M1 Format with Block-wise E8M0 Scales

# Build FP4 E2M1 lookup table
FP4_TABLE = np.zeros(16, dtype=np.float32)
for code in range(16):
    sign = -1.0 if (code >> 3) & 1 else 1.0
    exp_bits = (code >> 1) & 0x3
    mant_bit = code & 0x1
    if exp_bits == 0:
        value = 0.5 * mant_bit
    else:
        value = 2.0 ** (exp_bits - 1) * (1.0 + 0.5 * mant_bit)
    FP4_TABLE[code] = sign * value

_POS_VALUES = FP4_TABLE[:8]

def float_to_fp4(x):
    sign = 0
    if x < 0:
        sign = 8
        x = -x
    idx = int(np.argmin(np.abs(_POS_VALUES - x)))
    return sign | idx

float_to_fp4_vec = np.vectorize(float_to_fp4, otypes=[np.uint8])

def fp4_to_float(codes):
    return FP4_TABLE[codes]

def quantize_block_mx_fp4(block):
    amax = np.max(np.abs(block))
    if amax == 0:
        return 1.0, np.zeros(len(block), dtype=np.uint8)
    raw_exp = np.ceil(np.log2(amax / 6.0))
    scale = 2.0 ** raw_exp
    scaled = block / scale
    codes = float_to_fp4_vec(scaled)
    return scale, codes

def dequantize_block_mx_fp4(scale, codes):
    return scale * fp4_to_float(codes)

def pack_fp4(codes):
    n = len(codes)
    if n % 2 != 0:
        codes = np.append(codes, 0)
    packed = (codes[0::2] << 4) | codes[1::2]
    return packed.astype(np.uint8).tobytes()

def unpack_fp4(data, count):
    arr = np.frombuffer(data, dtype=np.uint8)
    hi = (arr >> 4) & 0xF
    lo = arr & 0xF
    codes = np.empty(len(arr) * 2, dtype=np.uint8)
    codes[0::2] = hi
    codes[1::2] = lo
    return codes[:count]

SKIP_KEYWORDS = {'bias', 'gamma', 'beta', 'moving_mean', 'moving_variance'}

def should_quantize(name):
    return all(kw not in name.lower() for kw in SKIP_KEYWORDS)

print("FP4 E2M1 values (positive):", _POS_VALUES)
print("MXFP4 codec ready.")

In [ ]:
#@title 6. Cross-Layer Equalization (CLE)
# Nagel et al., "Data-Free Quantization Through Weight Equalization" (2019)
# Rebalances weight ranges between consecutive conv layers:
#   scale_i = sqrt(max|W1[i,:]| / max|W2[:,i]|)
#   W1[i,:] /= scale_i, W2[:,i] *= scale_i

def cross_layer_equalization(model):
    """Apply CLE to consecutive conv layers."""

    if AIMET_AVAILABLE:
        from aimet_tensorflow.keras.cross_layer_equalization import equalize_model
        print("Applying AIMET Cross-Layer Equalization...")
        equalize_model(model, input_shape=INPUT_SHAPE)
        print("AIMET CLE complete.")
        return model

    print("Applying standalone Cross-Layer Equalization...")
    conv_layers = [l for l in model.layers
                   if isinstance(l, (tf.keras.layers.Conv2D,
                                     tf.keras.layers.DepthwiseConv2D,
                                     tf.keras.layers.Dense))]

    n_equalized = 0
    for i in range(len(conv_layers) - 1):
        layer1, layer2 = conv_layers[i], conv_layers[i + 1]

        # CLE applies to pointwise/regular conv pairs, skip depthwise
        if isinstance(layer1, tf.keras.layers.DepthwiseConv2D):
            continue
        if isinstance(layer2, tf.keras.layers.DepthwiseConv2D):
            continue

        w1, w2 = layer1.get_weights(), layer2.get_weights()
        if len(w1) == 0 or len(w2) == 0:
            continue

        kernel1, kernel2 = w1[0], w2[0]
        c_out1 = kernel1.shape[-1]
        if len(kernel2.shape) == 4:
            c_in2 = kernel2.shape[2]
        elif len(kernel2.shape) == 2:
            c_in2 = kernel2.shape[0]
        else:
            continue

        if c_out1 != c_in2:
            continue

        k1_flat = kernel1.reshape(-1, c_out1)
        if len(kernel2.shape) == 4:
            k2_flat = kernel2.transpose(2, 0, 1, 3).reshape(c_in2, -1)
        else:
            k2_flat = kernel2

        range1 = np.max(np.abs(k1_flat), axis=0) + 1e-10
        range2 = np.max(np.abs(k2_flat), axis=1) + 1e-10
        scale = np.clip(np.sqrt(range1 / range2), 0.01, 100.0)

        kernel1_new = kernel1 / scale.reshape([1] * (len(kernel1.shape) - 1) + [-1])
        if len(kernel2.shape) == 4:
            kernel2_new = kernel2 * scale.reshape(1, 1, -1, 1)
        else:
            kernel2_new = kernel2 * scale.reshape(-1, 1)

        w1_new = [kernel1_new] + ([w1[1] / scale] if len(w1) > 1 else [])
        w2_new = [kernel2_new] + list(w2[1:])
        layer1.set_weights(w1_new)
        layer2.set_weights(w2_new)
        n_equalized += 1
        print(f"  Equalized: {layer1.name} -> {layer2.name}  "
              f"(scale range: [{scale.min():.3f}, {scale.max():.3f}])")

    print(f"CLE complete: {n_equalized} layer pairs equalized.")
    return model

In [ ]:
#@title 7. Bias Correction
# Corrects systematic bias shift introduced by quantization.
# For each layer: bias_new = bias + E[conv(x, W)] - E[conv(x, W_q)]
# Approximated as: bias += sum_over_spatial(W - W_q) * mean_input_activation

def bias_correction(model, cal_data):
    """Correct quantization-induced bias shifts using calibration data."""

    if AIMET_AVAILABLE:
        from aimet_tensorflow.keras.bias_correction import BiasCorrection
        print("Applying AIMET Bias Correction...")
        def data_loader():
            for i in range(0, len(cal_data), 32):
                yield cal_data[i:i+32]
        BiasCorrection.correct_bias(model, data_loader)
        print("AIMET Bias Correction complete.")
        return model

    print("Applying standalone Bias Correction...")

    # Build intermediate-output extractors for empirical bias correction
    target_layers = [l for l in model.layers
                     if isinstance(l, (tf.keras.layers.Conv2D,
                                       tf.keras.layers.DepthwiseConv2D))
                     and l.use_bias]

    n_corrected = 0
    for layer in target_layers:
        weights = layer.get_weights()
        kernel, bias = weights[0], weights[1]

        # Simulate MXFP4 quantization of this layer's kernel
        flat = kernel.flatten()
        pad_len = (BLOCK_SIZE - len(flat) % BLOCK_SIZE) % BLOCK_SIZE
        padded = np.concatenate([flat, np.zeros(pad_len, dtype=np.float32)])
        q_flat = np.empty_like(padded)
        for b in range(len(padded) // BLOCK_SIZE):
            blk = padded[b * BLOCK_SIZE:(b + 1) * BLOCK_SIZE]
            scale, codes = quantize_block_mx_fp4(blk)
            q_flat[b * BLOCK_SIZE:(b + 1) * BLOCK_SIZE] = dequantize_block_mx_fp4(scale, codes)
        q_kernel = q_flat[:len(flat)].reshape(kernel.shape)

        # Empirical bias correction: run calibration data through the layer
        # with original vs quantized weights and correct the mean output shift.
        try:
            extractor = tf.keras.Model(inputs=model.input, outputs=layer.input)
            # Use a subset of calibration data for speed
            n_bc = min(64, len(cal_data))
            layer_in = extractor(cal_data[:n_bc], training=False).numpy()
        except Exception:
            # Fallback: analytical correction (sum of weight error per output channel)
            diff = kernel - q_kernel
            if isinstance(layer, tf.keras.layers.DepthwiseConv2D):
                # DepthwiseConv2D kernel: (kH, kW, C_in, depth_mult)
                # One bias per input channel
                bc = np.sum(diff, axis=(0, 1, 3))
            else:
                # Conv2D kernel: (kH, kW, C_in, C_out) — sum over spatial + input channels
                bc = np.sum(diff, axis=(0, 1, 2))
            bias_new = bias + bc
            correction_mag = np.mean(np.abs(bc))
            if correction_mag > 1e-8:
                layer.set_weights([kernel, bias_new])
                print(f"  Corrected {layer.name} (analytical): avg |shift| = {correction_mag:.6f}")
                n_corrected += 1
            continue

        # Compute output with original weights (no bias — we correct bias separately)
        if isinstance(layer, tf.keras.layers.Conv2D):
            out_orig = tf.nn.conv2d(layer_in, kernel, strides=layer.strides,
                                     padding=layer.padding.upper(),
                                     dilations=layer.dilation_rate).numpy()
            out_quant = tf.nn.conv2d(layer_in, q_kernel, strides=layer.strides,
                                      padding=layer.padding.upper(),
                                      dilations=layer.dilation_rate).numpy()
        elif isinstance(layer, tf.keras.layers.DepthwiseConv2D):
            out_orig = tf.nn.depthwise_conv2d(
                layer_in, kernel,
                strides=[1] + list(layer.strides) + [1],
                padding=layer.padding.upper(),
                dilations=layer.dilation_rate).numpy()
            out_quant = tf.nn.depthwise_conv2d(
                layer_in, q_kernel,
                strides=[1] + list(layer.strides) + [1],
                padding=layer.padding.upper(),
                dilations=layer.dilation_rate).numpy()

        # Mean output difference per channel = the bias shift to correct
        # Output shape: (N, H, W, C) — average over N, H, W
        bc = np.mean(out_orig - out_quant, axis=(0, 1, 2))
        bias_new = bias + bc
        correction_mag = np.mean(np.abs(bc))
        if correction_mag > 1e-8:
            layer.set_weights([kernel, bias_new])
            print(f"  Corrected {layer.name}: avg |shift| = {correction_mag:.6f}")
            n_corrected += 1

    print(f"Bias correction complete: {n_corrected} layers corrected.")
    return model

In [ ]:
#@title 8. AdaRound — Adaptive Rounding
# Nagel et al., "Up or Down? Adaptive Rounding for Post-Training Quantization" (ICML 2020)
# Learns per-weight floor/ceil rounding to minimize layer-wise reconstruction error.

import time as _time

def adaround_layer(layer, cal_inputs, n_iter=200, lr=1e-3):
    """Apply AdaRound to a single layer using @tf.function-compiled training."""
    weights = layer.get_weights()
    kernel = weights[0].astype(np.float32)
    flat = kernel.flatten()

    pad_len = (BLOCK_SIZE - len(flat) % BLOCK_SIZE) % BLOCK_SIZE
    padded = np.concatenate([flat, np.zeros(pad_len, dtype=np.float32)])
    n_blocks = len(padded) // BLOCK_SIZE

    scales = []
    for b in range(n_blocks):
        blk = padded[b * BLOCK_SIZE:(b + 1) * BLOCK_SIZE]
        amax = np.max(np.abs(blk))
        scales.append(2.0 ** np.ceil(np.log2(amax / 6.0)) if amax > 0 else 1.0)
    scales = np.array(scales, dtype=np.float32)
    elem_scales = np.repeat(scales, BLOCK_SIZE)[:len(padded)]
    scaled_w = padded / elem_scales

    # Find floor/ceil FP4 values for each weight
    all_fp4 = np.unique(np.concatenate([-_POS_VALUES[::-1], _POS_VALUES]))
    floor_vals = np.zeros_like(scaled_w)
    ceil_vals = np.zeros_like(scaled_w)
    for i, sw in enumerate(scaled_w):
        diffs = all_fp4 - sw
        below = all_fp4[diffs <= 0]
        above = all_fp4[diffs >= 0]
        floor_vals[i] = below[-1] if len(below) > 0 else all_fp4[0]
        ceil_vals[i] = above[0] if len(above) > 0 else all_fp4[-1]

    needs_rounding = (floor_vals != ceil_vals)
    if not np.any(needs_rounding):
        return kernel

    # Initialize v so sigmoid(v) ≈ fractional position between floor and ceil
    # This gives the optimizer a sensible starting point (near-nearest rounding)
    round_idx = np.where(needs_rounding)[0]
    frac = (scaled_w[round_idx] - floor_vals[round_idx]) / \
           np.maximum(ceil_vals[round_idx] - floor_vals[round_idx], 1e-10)
    frac = np.clip(frac, 0.01, 0.99)
    v_init = -np.log(1.0 / frac - 1.0)  # inverse sigmoid
    v = tf.Variable(v_init.astype(np.float32))
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

    n_cal = min(32, len(cal_inputs))
    cal_batch = tf.constant(cal_inputs[:n_cal], dtype=tf.float32)

    if isinstance(layer, tf.keras.layers.Conv2D):
        _strides = layer.strides
        _padding = layer.padding.upper()
        _dilations = layer.dilation_rate
        conv_fn = lambda k: tf.nn.conv2d(cal_batch, k, strides=_strides,
                                          padding=_padding, dilations=_dilations)
    elif isinstance(layer, tf.keras.layers.DepthwiseConv2D):
        _strides = [1] + list(layer.strides) + [1]
        _padding = layer.padding.upper()
        _dilations = layer.dilation_rate
        conv_fn = lambda k: tf.nn.depthwise_conv2d(cal_batch, k, strides=_strides,
                                                     padding=_padding, dilations=_dilations)
    elif isinstance(layer, tf.keras.layers.Dense):
        conv_fn = lambda k: tf.matmul(cal_batch, k)
    else:
        return kernel

    ref_output = conv_fn(tf.constant(kernel))
    floor_r = tf.constant(floor_vals[round_idx], dtype=tf.float32)
    ceil_r = tf.constant(ceil_vals[round_idx], dtype=tf.float32)
    elem_scales_r = tf.constant(elem_scales[round_idx], dtype=tf.float32)
    fixed_q = floor_vals.copy()
    fixed_q_scaled = (fixed_q * elem_scales)[:len(flat)]
    indices = tf.constant(round_idx.astype(np.int32).reshape(-1, 1))
    fixed_q_scaled_t = tf.constant(fixed_q_scaled, dtype=tf.float32)
    kernel_shape = kernel.shape

    # Use a tf.Variable for beta so @tf.function doesn't retrace each step
    beta_var = tf.Variable(2.0, dtype=tf.float32, trainable=False)

    @tf.function(reduce_retracing=True)
    def train_step():
        with tf.GradientTape() as tape:
            h = tf.clip_by_value(tf.sigmoid(beta_var * v), 0.0, 1.0)
            q_r = (floor_r + h * (ceil_r - floor_r)) * elem_scales_r
            q_all = tf.tensor_scatter_nd_update(fixed_q_scaled_t, indices, q_r)
            q_kernel = tf.reshape(q_all, kernel_shape)
            q_output = conv_fn(q_kernel)
            rec_loss = tf.reduce_mean((ref_output - q_output) ** 2)
            reg_loss = 0.01 * tf.reduce_mean(h * (1.0 - h))
            loss = rec_loss + reg_loss
        grads = tape.gradient(loss, [v])
        if grads[0] is not None:
            optimizer.apply_gradients(zip(grads, [v]))
        return loss

    best_loss, best_v = float('inf'), None
    t0 = _time.time()

    for step in range(n_iter):
        progress = step / max(n_iter - 1, 1)
        # Anneal beta: soft (2) → hard (20) per AdaRound paper
        beta_var.assign(2.0 + progress * (20.0 - 2.0))
        l = train_step().numpy()
        if l < best_loss:
            best_loss = l
            best_v = v.numpy().copy()
        if (step + 1) % 100 == 0:
            print(".", end="", flush=True)

    # Final hard rounding at beta=20
    h_final = (1.0 / (1.0 + np.exp(-20.0 * best_v))) > 0.5
    q_final = fixed_q.copy()
    q_final[round_idx[h_final]] = ceil_vals[round_idx[h_final]]
    q_final[round_idx[~h_final]] = floor_vals[round_idx[~h_final]]
    elapsed = _time.time() - t0
    print(f" loss={best_loss:.6f} ({elapsed:.1f}s)", end="", flush=True)
    return ((q_final * elem_scales)[:len(flat)]).reshape(kernel.shape)


def apply_adaround(model, cal_data, n_iter=200):
    """Apply AdaRound to all conv/dense layers."""

    if AIMET_AVAILABLE:
        from aimet_tensorflow.keras.adaround_weight import Adaround, AdaroundParameters
        print("Applying AIMET AdaRound...")
        def data_loader():
            for i in range(0, len(cal_data), 32):
                yield cal_data[i:i+32]
        params = AdaroundParameters(data_loader, num_batches=len(cal_data)//32)
        Adaround.apply_adaround(model, params)
        print("AIMET AdaRound complete.")
        return model

    print(f"Applying standalone AdaRound ({n_iter} iters/layer, @tf.function compiled)...")

    # Suppress TF retracing warnings during the extraction loop
    import logging
    tf_logger = tf.get_logger()
    prev_level = tf_logger.level
    tf_logger.setLevel(logging.ERROR)

    target_layers = [l for l in model.layers
                     if isinstance(l, (tf.keras.layers.Conv2D,
                                       tf.keras.layers.DepthwiseConv2D,
                                       tf.keras.layers.Dense))
                     and should_quantize(l.name)]

    # Pre-build all extractors at once to reduce retracing overhead
    extractors = {}
    for layer in target_layers:
        try:
            if hasattr(layer, 'input') and layer.input is not None:
                extractors[layer.name] = tf.keras.Model(
                    inputs=model.input, outputs=layer.input)
        except Exception:
            pass

    for i, layer in enumerate(target_layers):
        print(f"  [{i+1}/{len(target_layers)}] {layer.name} ", end="", flush=True)
        try:
            if layer.name in extractors:
                layer_inputs = extractors[layer.name](cal_data, training=False).numpy()
            else:
                layer_inputs = cal_data
        except Exception:
            print("(skipped)")
            continue

        optimized_kernel = adaround_layer(layer, layer_inputs, n_iter=n_iter)
        w = layer.get_weights()
        w[0] = optimized_kernel
        layer.set_weights(w)
        print(f" shape={optimized_kernel.shape}")

    tf_logger.setLevel(prev_level)
    print(f"AdaRound complete: {len(target_layers)} layers optimized.")
    return model

In [ ]:
#@title 9. Run AIMET Pipeline: CLE → Bias Correction → AdaRound

print("="*60)
print("AIMET Optimization Pipeline")
print("="*60)

# Start from a fresh copy of the Keras model
model_aimet = tf.keras.models.load_model(h5_model_path, compile=False)

# Step 1: Cross-Layer Equalization
print("\n--- Step 1/3: Cross-Layer Equalization ---")
model_aimet = cross_layer_equalization(model_aimet)

# Step 2: Bias Correction
print("\n--- Step 2/3: Bias Correction ---")
model_aimet = bias_correction(model_aimet, cal_data)

# Step 3: AdaRound
print("\n--- Step 3/3: AdaRound ---")
model_aimet = apply_adaround(model_aimet, cal_data, n_iter=adaround_iters)

print("\nAIMET pipeline complete.")

In [ ]:
#@title 10. MXFP4 Quantize — Baseline vs AIMET-Enhanced

def quantize_model_to_mxfp4(model, label=""):
    """Quantize all kernel weights to MXFP4."""
    print(f"\nQuantizing to MXFP4 [{label}]...")
    entries = []
    total_orig, total_fp4, total_kept = 0, 0, 0

    for idx, w in enumerate(model.weights):
        name = f"w{idx}_{w.name}"
        arr = w.numpy().flatten()
        shape = w.shape
        total_orig += arr.nbytes

        if should_quantize(w.name):
            pad_len = (BLOCK_SIZE - len(arr) % BLOCK_SIZE) % BLOCK_SIZE
            padded = np.concatenate([arr, np.zeros(pad_len, dtype=np.float32)])
            all_scales, all_codes = [], []
            for b in range(len(padded) // BLOCK_SIZE):
                block = padded[b * BLOCK_SIZE:(b + 1) * BLOCK_SIZE]
                scale, codes = quantize_block_mx_fp4(block)
                all_scales.append(scale)
                all_codes.append(codes)

            scales = np.array(all_scales, dtype=np.float32)
            codes = np.concatenate(all_codes)[:len(arr)]
            total_fp4 += scales.nbytes + (len(arr) + 1) // 2
            entries.append({'name': name, 'shape': list(shape),
                            'n_elements': len(arr), 'quantized': True,
                            'scales': scales, 'codes': codes})
        else:
            total_kept += arr.nbytes
            entries.append({'name': name, 'shape': list(shape),
                            'n_elements': len(arr), 'quantized': False,
                            'raw_data': arr.astype(np.float32)})

    total_comp = total_fp4 + total_kept
    print(f"  Original: {total_orig:,} B | FP4: {total_fp4:,} B | "
          f"BN/bias: {total_kept:,} B | Total: {total_comp:,} B | "
          f"Ratio: {total_orig/total_comp:.1f}x")
    return entries

def dequantize_entry(entry):
    """Reconstruct float32 weights from an FP4 entry."""
    if not entry.get('quantized', True):
        return entry['raw_data'].reshape(entry['shape'])
    n = entry['n_elements']
    scales, codes = entry['scales'], entry['codes']
    pad_len = (BLOCK_SIZE - n % BLOCK_SIZE) % BLOCK_SIZE
    codes_p = np.concatenate([codes, np.zeros(pad_len, dtype=np.uint8)])
    result = np.empty(len(codes_p), dtype=np.float32)
    for b in range(len(codes_p) // BLOCK_SIZE):
        s = scales[b]
        c = codes_p[b*BLOCK_SIZE:(b+1)*BLOCK_SIZE]
        result[b*BLOCK_SIZE:(b+1)*BLOCK_SIZE] = dequantize_block_mx_fp4(s, c)
    return result[:n].reshape(entry['shape'])

def apply_fp4_weights(model, entries):
    """Load dequantized FP4 weights into a Keras model."""
    model.set_weights([dequantize_entry(e) for e in entries])

# Quantize baseline (nearest rounding, no AIMET)
model_baseline = tf.keras.models.load_model(h5_model_path, compile=False)
entries_baseline = quantize_model_to_mxfp4(model_baseline, "Baseline (nearest rounding)")

# Quantize AIMET-enhanced model
entries_aimet = quantize_model_to_mxfp4(model_aimet, "AIMET CLE + BC + AdaRound")

In [ ]:
#@title 11. Save FP4 Binary & Convert to int8 TFLite

MAGIC = b'MXF4'
FORMAT_VERSION = 2

def save_fp4_binary(entries, path):
    with open(path, 'wb') as f:
        f.write(MAGIC)
        f.write(struct.pack('<B', FORMAT_VERSION))
        f.write(struct.pack('<I', len(entries)))
        for e in entries:
            quantized = e.get('quantized', True)
            f.write(struct.pack('<B', 1 if quantized else 0))
            name_bytes = e['name'].encode('utf-8')
            f.write(struct.pack('<H', len(name_bytes)))
            f.write(name_bytes)
            shape = e['shape']
            f.write(struct.pack('<B', len(shape)))
            for s in shape:
                f.write(struct.pack('<I', s))
            f.write(struct.pack('<I', e['n_elements']))
            if quantized:
                f.write(struct.pack('<I', len(e['scales'])))
                f.write(e['scales'].tobytes())
                packed = pack_fp4(e['codes'])
                f.write(struct.pack('<I', len(packed)))
                f.write(packed)
            else:
                f.write(e['raw_data'].astype(np.float32).tobytes())
    return os.path.getsize(path)

def convert_to_int8_tflite(entries, cal_data, output_path):
    """Apply FP4 weights to a Keras model and convert to int8 TFLite."""
    m = tf.keras.models.load_model(h5_model_path, compile=False)
    apply_fp4_weights(m, entries)

    def rep_data():
        for i in range(min(n_calibration, len(cal_data))):
            yield [cal_data[i:i+1]]

    converter = tf.lite.TFLiteConverter.from_keras_model(m)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = rep_data
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    tflite_bytes = converter.convert()

    with open(output_path, 'wb') as f:
        f.write(tflite_bytes)
    return len(tflite_bytes)

os.makedirs('aimet_output', exist_ok=True)

# Save baseline FP4
baseline_fp4_path = 'aimet_output/model_baseline_mxfp4.fp4bin'
baseline_fp4_size = save_fp4_binary(entries_baseline, baseline_fp4_path)

# Save AIMET FP4
aimet_fp4_path = 'aimet_output/model_aimet_mxfp4.fp4bin'
aimet_fp4_size = save_fp4_binary(entries_aimet, aimet_fp4_path)

# Convert both to int8 TFLite
print("Converting Baseline FP4 -> int8 TFLite...")
baseline_tflite_path = 'aimet_output/model_baseline_mxfp4_int8.tflite'
baseline_tfl_size = convert_to_int8_tflite(entries_baseline, cal_data, baseline_tflite_path)

print("Converting AIMET FP4 -> int8 TFLite...")
aimet_tflite_path = 'aimet_output/model_aimet_mxfp4_int8.tflite'
aimet_tfl_size = convert_to_int8_tflite(entries_aimet, cal_data, aimet_tflite_path)

print(f"\nSaved:")
print(f"  Baseline FP4:       {baseline_fp4_path}  ({baseline_fp4_size/1024:.1f} KB)")
print(f"  AIMET FP4:          {aimet_fp4_path}  ({aimet_fp4_size/1024:.1f} KB)")
print(f"  Baseline int8 TFL:  {baseline_tflite_path}  ({baseline_tfl_size/1024:.1f} KB)")
print(f"  AIMET int8 TFL:     {aimet_tflite_path}  ({aimet_tfl_size/1024:.1f} KB)")

In [ ]:
#@title 12. Load COCO Minival Evaluation Set

def resize2SquareKeepingAspectRatio(img, size, interpolation):
    h, w = img.shape[:2]
    c = None if len(img.shape) < 3 else img.shape[2]
    dif = max(h, w)
    x_pos = int((dif - w) / 2.)
    y_pos = int((dif - h) / 2.)
    if c is None:
        mask = np.zeros((dif, dif), dtype=img.dtype)
        mask[y_pos:y_pos+h, x_pos:x_pos+w] = img[:h, :w]
    else:
        mask = np.zeros((dif, dif, c), dtype=img.dtype)
        mask[y_pos:y_pos+h, x_pos:x_pos+w, :] = img[:h, :w, :]
    return cv2.resize(mask, (size, size), interpolation)

def load_minival(data_dir, minival_path):
    """Load COCO minival set with person/non-person labels."""
    dataType = 'val2014'
    annFile = os.path.join(data_dir, 'annotations', f'instances_{dataType}.json')
    print(f"Loading COCO annotations from {annFile}...")
    coco = COCO(annFile)

    catIds = coco.getCatIds(catNms=['person'])
    imgIds = coco.getImgIds(catIds=catIds)
    imgMVIds = [int(x) for x in open(minival_path)]
    personImgIds = list(set(imgMVIds) & set(imgIds))
    nonPersonImgIds = list(set(imgMVIds) - set(imgIds))

    # Count valid person images (area > threshold)
    numValidPersons = 0
    for imgId in personImgIds:
        img = coco.loadImgs(imgId)[0]
        annIds = coco.getAnnIds(imgIds=img['id'], catIds=catIds, iscrowd=0)
        anns = coco.loadAnns(annIds)
        if any(ann['area'] > PERSONTHR for ann in anns):
            numValidPersons += 1

    total = numValidPersons + len(nonPersonImgIds)
    print(f"Person images (area>{PERSONTHR}): {numValidPersons}")
    print(f"Non-person images: {len(nonPersonImgIds)}")
    print(f"Total eval images: {total}")

    images = np.ones((total, IMAGESIZE, IMAGESIZE, 3), np.uint8)
    labels = np.zeros((total, 1), np.uint8)
    k = 0

    # Person images
    for imgId in personImgIds:
        img = coco.loadImgs(imgId)[0]
        annIds = coco.getAnnIds(imgIds=img['id'], catIds=catIds, iscrowd=0)
        anns = coco.loadAnns(annIds)
        if not any(ann['area'] > PERSONTHR for ann in anns):
            continue
        name = os.path.join(data_dir, dataType, img['file_name'])
        if not os.path.exists(name):
            continue
        img1 = cv2.imread(name)
        if img1 is None:
            continue
        img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
        if len(img1.shape) != 3:
            img1 = cv2.cvtColor(img1, cv2.COLOR_GRAY2RGB)
        images[k] = resize2SquareKeepingAspectRatio(img1, IMAGESIZE, cv2.INTER_AREA)
        labels[k] = 1
        k += 1

    person_count = k

    # Non-person images
    for imgId in nonPersonImgIds:
        img = coco.loadImgs(imgId)[0]
        name = os.path.join(data_dir, dataType, img['file_name'])
        if not os.path.exists(name):
            continue
        img1 = cv2.imread(name)
        if img1 is None:
            continue
        img1 = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)
        if len(img1.shape) != 3:
            img1 = cv2.cvtColor(img1, cv2.COLOR_GRAY2RGB)
        images[k] = resize2SquareKeepingAspectRatio(img1, IMAGESIZE, cv2.INTER_AREA)
        labels[k] = 0
        k += 1

    images = images[:k]
    labels = labels[:k]

    # Shuffle deterministically
    idx = np.arange(k)
    np.random.seed(42)
    np.random.shuffle(idx)
    images = images[idx]
    labels = labels[idx]

    # Normalize to [-1, 1]
    images = images.astype('float32') / 127.5 - 1.0

    print(f"Loaded {k} images ({person_count} person, {k - person_count} non-person)")
    return images, labels

# Create symlink structure for data_dir
data_dir = '/content/coco_eval'
os.makedirs(f'{data_dir}/val2014', exist_ok=True)
os.makedirs(f'{data_dir}/annotations', exist_ok=True)

# Symlink (no copy)
for name, src in [('val2014', val2014_folder), ('annotations', annotations_folder)]:
    dst = f'{data_dir}/{name}'
    if os.path.islink(dst) or os.path.isdir(dst):
        !rm -rf {dst}
    os.symlink(src, dst)

x_test, y_test = load_minival(data_dir, minival_path)
print(f"\nEval set: {x_test.shape[0]} images, {int(np.sum(y_test))} person, "
      f"{x_test.shape[0] - int(np.sum(y_test))} non-person")

In [ ]:
#@title 13. Evaluate All Model Variants on COCO Minival

def evaluate_keras(model, x_test, y_test, name="Model"):
    """Evaluate a Keras model."""
    preds = model.predict(x_test, batch_size=64, verbose=0)
    pred_labels = (preds > 0.5).astype(np.uint8)
    correct = np.sum(pred_labels == y_test)
    acc = correct / len(y_test) * 100
    print(f"  {name}: {correct}/{len(y_test)} = {acc:.2f}%")
    return acc

def evaluate_tflite(tflite_path, x_test, y_test, name="TFLite"):
    """Evaluate a TFLite model."""
    interp = tf.lite.Interpreter(model_path=tflite_path)
    interp.allocate_tensors()
    inp = interp.get_input_details()
    out = interp.get_output_details()

    correct = 0
    total = len(y_test)
    for i in range(total):
        img = x_test[i:i+1]
        if inp[0]['dtype'] == np.int8:
            s = inp[0]['quantization_parameters']['scales'][0]
            z = inp[0]['quantization_parameters']['zero_points'][0]
            img = (img / s + z).astype(np.int8)

        interp.set_tensor(inp[0]['index'], img)
        interp.invoke()
        output = interp.get_tensor(out[0]['index'])

        if out[0]['dtype'] == np.int8:
            s = out[0]['quantization_parameters']['scales'][0]
            z = out[0]['quantization_parameters']['zero_points'][0]
            output = (output.astype(np.float32) - z) * s

        pred = 1 if output[0] > 0.5 else 0
        if pred == y_test[i]:
            correct += 1

        if (i + 1) % 1000 == 0:
            print(f"    {name}: {i+1}/{total}, running acc={correct/(i+1)*100:.1f}%")

    acc = correct / total * 100
    print(f"  {name}: {correct}/{total} = {acc:.2f}%")
    return acc

results = []

# 1. Original Keras H5
print("="*60)
print("[1/6] Original Keras H5 model")
m = tf.keras.models.load_model(h5_model_path, compile=False)
acc = evaluate_keras(m, x_test, y_test, "Original H5")
results.append(("Original H5", acc, h5_size))

# 2. Original TFLite
print("\n[2/6] Original TFLite model")
acc = evaluate_tflite(tflite_model_path, x_test, y_test, "Original TFLite")
results.append(("Original TFLite", acc, tflite_size))

# 3. Baseline MXFP4 (dequantized in Keras)
print("\n[3/6] Baseline MXFP4 (nearest rounding, no AIMET)")
m_bl = tf.keras.models.load_model(h5_model_path, compile=False)
apply_fp4_weights(m_bl, entries_baseline)
acc = evaluate_keras(m_bl, x_test, y_test, "Baseline FP4 (Keras)")
results.append(("Baseline MXFP4", acc, baseline_fp4_size))

# 4. AIMET MXFP4 (dequantized in Keras)
print("\n[4/6] AIMET-enhanced MXFP4 (CLE + BC + AdaRound)")
m_ai = tf.keras.models.load_model(h5_model_path, compile=False)
apply_fp4_weights(m_ai, entries_aimet)
acc = evaluate_keras(m_ai, x_test, y_test, "AIMET FP4 (Keras)")
results.append(("AIMET MXFP4", acc, aimet_fp4_size))

# 5. Baseline FP4 → int8 TFLite
print("\n[5/6] Baseline MXFP4 -> int8 TFLite")
acc = evaluate_tflite(baseline_tflite_path, x_test, y_test, "Baseline FP4->int8 TFL")
results.append(("Baseline FP4->int8 TFL", acc, baseline_tfl_size))

# 6. AIMET FP4 → int8 TFLite
print("\n[6/6] AIMET MXFP4 -> int8 TFLite")
acc = evaluate_tflite(aimet_tflite_path, x_test, y_test, "AIMET FP4->int8 TFL")
results.append(("AIMET FP4->int8 TFL", acc, aimet_tfl_size))

print("\nAll evaluations complete.")

In [ ]:
#@title 14. Final Results Summary

print(f"{'='*70}")
print("RESULTS: Visual Wake Word — COCO Minival Accuracy")
print(f"{'='*70}")
print(f"  {'Model':<30} {'Accuracy':>8}  {'Size':>8}  {'vs H5':>8}")
print(f"  {'-'*30} {'-'*8}  {'-'*8}  {'-'*8}")

baseline_acc = results[0][1]  # Original H5 accuracy
for name, acc, size in results:
    delta = acc - baseline_acc
    delta_str = f"{delta:+.2f}%" if name != "Original H5" else "   ref"
    size_kb = f"{size/1024:.0f} KB"
    print(f"  {name:<30} {acc:>7.2f}%  {size_kb:>8}  {delta_str:>8}")

print(f"\n  AIMET improvement over baseline FP4:")
bl_fp4_acc = results[2][1]  # Baseline MXFP4
ai_fp4_acc = results[3][1]  # AIMET MXFP4
print(f"    Baseline MXFP4:  {bl_fp4_acc:.2f}%")
print(f"    AIMET MXFP4:     {ai_fp4_acc:.2f}%")
print(f"    Delta:           {ai_fp4_acc - bl_fp4_acc:+.2f}%")

if len(results) >= 6:
    bl_tfl_acc = results[4][1]
    ai_tfl_acc = results[5][1]
    print(f"\n    Baseline FP4->int8 TFL:  {bl_tfl_acc:.2f}%")
    print(f"    AIMET FP4->int8 TFL:     {ai_tfl_acc:.2f}%")
    print(f"    Delta:                   {ai_tfl_acc - bl_tfl_acc:+.2f}%")

print(f"{'='*70}")

In [ ]:
#@title 15. (Optional) Download Output Files
from google.colab import files

print("Output files:")
for f in sorted(os.listdir('aimet_output')):
    size = os.path.getsize(os.path.join('aimet_output', f))
    print(f"  {f}: {size:,} B ({size/1024:.1f} KB)")

# Uncomment to download:
# files.download('aimet_output/model_aimet_mxfp4.fp4bin')
# files.download('aimet_output/model_aimet_mxfp4_int8.tflite')